In [23]:
import pandas as pd

In [24]:
df=pd.read_csv('/content/WA_Fn-UseC_-HR-Employee-Attrition.csv')

In [25]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [26]:
df=df.drop('Over18',axis=1)

In [27]:
df['Attrition']=df['Attrition'].map({'Yes':1,'No':0})
df['Gender']=df['Gender'].map({'Male':1,'Female':0})
df['OverTime']=df['OverTime'].map({'Yes':1,'No':0})

In [28]:
x=df.drop('Attrition',axis=1)
y=df['Attrition']

In [29]:
cat_cols=['BusinessTravel','Department','EducationField','MaritalStatus','JobRole']
num_cols=x.select_dtypes(include=['int64','float64']).columns

In [30]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.30,random_state=13)

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler

ct = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(), cat_cols)
    ],
    remainder='passthrough'
).set_output(transform='pandas')

In [32]:
train_encoded=ct.fit_transform(x_train)
test_encoded=ct.transform(x_test)

In [33]:
ss=StandardScaler()

train_encoded_scaled=ss.fit_transform(train_encoded)
test_encoded_scaled=ss.transform(test_encoded)

train_encoded_scaled = pd.DataFrame(
    train_encoded_scaled,
    columns=train_encoded.columns,
    index=train_encoded.index
)

test_encoded_scaled = pd.DataFrame(
    test_encoded_scaled,
    columns=test_encoded.columns,
    index=test_encoded.index
)

In [34]:
train_encoded_scaled.head()

,cat__BusinessTravel,cat__Department,cat__EducationField,cat__MaritalStatus,cat__JobRole,remainder__Age,remainder__DailyRate,remainder__DistanceFromHome,remainder__Education,remainder__EmployeeCount,...,remainder__RelationshipSatisfaction,remainder__StandardHours,remainder__StockOptionLevel,remainder__TotalWorkingYears,remainder__TrainingTimesLastYear,remainder__WorkLifeBalance,remainder__YearsAtCompany,remainder__YearsInCurrentRole,remainder__YearsSinceLastPromotion,remainder__YearsWithCurrManager
924,0.576956,-0.490473,-0.940647,-0.141536,0.601934,-0.193959,-0.169368,-0.417476,-1.879828,0.0,...,1.229090,0.0,-0.923505,-0.925017,0.139558,0.333150,-0.631056,-0.595840,-0.037675,-0.579360
986,0.576956,1.344787,-0.940647,-0.141536,1.006236,0.240627,1.744502,1.400735,1.049515,0.0,...,1.229090,0.0,1.380221,-0.405311,-0.636684,1.755605,-0.306783,-0.041733,-0.352859,-0.013742
1263,0.576956,-0.490473,0.578860,-1.515506,-1.015273,0.566566,0.131633,0.309809,0.073068,0.0,...,-0.618585,0.0,2.532084,-0.535237,2.468282,-1.089304,-0.306783,-0.318787,-0.668042,-0.013742
660,-0.933700,-0.490473,-0.940647,-1.515506,-1.015273,2.304909,-0.053984,-0.902332,-1.879828,0.0,...,1.229090,0.0,0.228358,-1.054944,0.139558,-1.089304,-0.955328,-1.149947,-0.668042,-1.144978
976,0.576956,-0.490473,-0.940647,-0.141536,-0.206669,2.087616,1.420925,1.643164,0.073068,0.0,...,-1.542423,0.0,0.228358,2.842857,-2.189167,0.333150,1.963127,3.282910,4.059711,1.400303


In [35]:
train_encoded_scaled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1029 entries, 924 to 338
Data columns (total 33 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   cat__BusinessTravel                  1029 non-null   float64
 1   cat__Department                      1029 non-null   float64
 2   cat__EducationField                  1029 non-null   float64
 3   cat__MaritalStatus                   1029 non-null   float64
 4   cat__JobRole                         1029 non-null   float64
 5   remainder__Age                       1029 non-null   float64
 6   remainder__DailyRate                 1029 non-null   float64
 7   remainder__DistanceFromHome          1029 non-null   float64
 8   remainder__Education                 1029 non-null   float64
 9   remainder__EmployeeCount             1029 non-null   float64
 10  remainder__EmployeeNumber            1029 non-null   float64
 11  remainder__EnvironmentSatisfaction

In [36]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [37]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(
            features.values,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels.values,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        return self.features[idx], self.labels[idx]

In [38]:
train_dataset=CustomDataset(train_encoded_scaled,y_train)
test_dataset=CustomDataset(test_encoded_scaled,y_test)

In [39]:
train_loader=DataLoader(train_dataset,batch_size=90,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=90,shuffle=True)

In [40]:
feature_count=len(train_encoded_scaled.columns)
print(feature_count)

33


In [41]:
class MyNN(nn.Module):

  def __init__(self,feature_count):

    super().__init__()

    self.network=nn.Sequential(
        nn.Linear(feature_count,66),
        nn.ReLU(),

        nn.Linear(66,33),
        nn.ReLU(),

        nn.Linear(33,1),
  )


  def forward(self,features):
    self.features=features

    return(self.network(features))


In [42]:
model=MyNN(feature_count=feature_count)

In [43]:
criterion=nn.BCEWithLogitsLoss()

optimizer=torch.optim.Adam(params=model.parameters(),lr=0.001)

In [44]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for features, labels in train_loader:

        labels = labels.unsqueeze(1)

        outputs = model(features)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Epoch 1 Loss: 7.0032
Epoch 2 Loss: 6.0884
Epoch 3 Loss: 5.2557
Epoch 4 Loss: 5.0139
Epoch 5 Loss: 4.6160
Epoch 6 Loss: 4.5417
Epoch 7 Loss: 4.3905
Epoch 8 Loss: 4.1973
Epoch 9 Loss: 4.1025
Epoch 10 Loss: 3.8958
Epoch 11 Loss: 3.7536
Epoch 12 Loss: 3.7088
Epoch 13 Loss: 3.5234
Epoch 14 Loss: 3.3848
Epoch 15 Loss: 3.2351
Epoch 16 Loss: 3.2414
Epoch 17 Loss: 3.1045
Epoch 18 Loss: 3.0505
Epoch 19 Loss: 2.9571
Epoch 20 Loss: 3.0171


In [45]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for features, labels in test_loader:

        labels = labels.unsqueeze(1)

        outputs = model(features)

        predictions = (
            torch.sigmoid(outputs) > 0.5
        ).float()

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

accuracy = correct / total

print("Accuracy:", accuracy)

Accuracy: 0.873015873015873
